##  Introduction to Spike Data Analysis

Up to this point, we have focused on analyzing **local field potentials (LFPs)**.  
Another essential type of neural signal is **spike data** — the rapid voltage fluctuations produced by individual neurons when they fire action potentials.

Modern electrophysiology tools (e.g., Neuropixels probes, Utah arrays, tetrodes, multi-electrode arrays) allow us to record the activity of **dozens to thousands of neurons simultaneously**. After preprocessing and spike sorting, spike data typically takes the form of **timestamps** indicating when each neuron fired.

One major advantage of spike recordings is their **spatial specificity**:  
we can often attribute spikes to single neurons, and even in multi-unit recordings, we know the activity originates from neurons located **very close to the electrode tip**.

In this notebook we'll analyze some spike data. We'll first extract the spikes from the raw data. 
I'll also briefly showcase how you could attribute your spikes to individual neurons. Nowadays there are advanced algorithms 
(Kilosort, mountainsort etc) that can do this more accurately and effictively but our simple example will still showcase the general idea.


---

##  What Does Extracellular Data Look Like?

The electrode measures a **voltage difference** between the electrode tip and a reference electrode.  
This raw extracellular signal contains a mixture of components:

- **Low-frequency population activity (LFP)**  
- **High-frequency action potentials (spikes)**

Why are spikes high-frequency?

- Action potentials rise and fall extremely fast (~0.5–1 ms).  
- Fast events contain high-frequency energy.  
- Therefore, spikes dominate the **300–3000 Hz** range, while LFP signals reside below **~200 Hz**.

We can extract these components using band-pass filters:

- **LFP band:**  
  $$0 \text{–} 200~\text{Hz}$$

- **Spike (MUA/SUA) band:**  
  $$300 \text{–} 3000~\text{Hz}$$

This simple filtering step already provides a powerful separation between slow population dynamics and fast spiking activity.

---

##  LFP vs MUA vs SUA — What’s the Difference?

- **LFP (Local Field Potential):**  
  Low-frequency activity reflecting summed synaptic currents from many neurons.

- **MUA (Multi-Unit Activity):**  
  High-frequency components reflecting the combined spiking of several nearby neurons.

- **SUA (Single-Unit Activity):**  
  Spikes that are attributed to individual neurons after *spike sorting*.

Understanding this distinction is crucial, as each signal reflects different spatial and temporal scales of neural computation.

---

Okay let's try it out! In this notebook we'll analyze the Spike_data_1.npz dataset. 
This dataset contains raw electrophysiology data in which we can find the spiking activity of a couple of neurons. 
The data was collected during an experiment in which we presented a super brief auditory stimulus every second. 


In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import scipy as sp 

Raw_data = np.load('Spike_data_1.npz')


Question 1: What type of variable is the data that we load in? What does it contain? 

Answer: 


Question 2: Plot the first 5s of the raw signal 

In [ ]:
# Your code here


**Question 3 — Filtering the Raw Signal to Extract Spikes**

We are now interested in extracting **spikes** from our raw neural recording.  
To do this, we first need to **filter the raw voltage trace** to isolate the frequency band in which spikes occur.

Extracellular spikes typically lie in the range of approximately **300–5000 Hz**, so applying a band-pass filter in this range will help suppress slow components (like local field potentials) and very fast noise.


### **Your Tasks**

1. **Choose an appropriate filter**  
   Use your knowledge of digital filters (e.g., FIR or IIR filters).  
   A band-pass filter around the spike frequency range is recommended.

2. **Apply the filter to the raw voltage signal**  


3. **Plot the filtered signal**  

4. **Adjust the time window if needed**  
   Spikes are brief events (≈1 ms), so a zoomed-in time window will make them easier to see.  
   Try plotting only a small segment (e.g., 0.1 seconds or a short trial window).

5. **Describe what you observe**  
   - Do you see brief, sharp deflections?  
   - Does the filter remove slow drifts and noise?  
   - Are the spikes easier to distinguish now?



In [ ]:
# Your code here: design and apply a band-pass filter (300-5000 Hz)


**Question 4 — Spike Detection Using a Threshold**

In the previous question, you should have observed that the raw data clearly contains **action potentials (spikes)**. Our next goal is to **extract these spike events** from the signal.

A simple way to detect spikes is to find all time points where the signal exceeds a certain **threshold**. Choosing this threshold is somewhat arbitrary:

- If the threshold is **too low**, noise fluctuations may be falsely classified as spikes.  
- If the threshold is **too high**, true spikes may be missed.

A commonly used automatic threshold is:

$ \text{Threshold} = \mu \pm k \cdot \sigma $

Where:

- $ \mu $ = mean of the signal  
- $ \sigma $ = standard deviation  
- $ k $ = scaling factor (try values such as **2**, **2.5**, **3**)

**Your Task**

1. Compute thresholds for multiple values of $k$ (e.g., **2**, **2.5**, **3**).  
2. Overlay these thresholds on the raw signal plot.  
3. Decide which threshold is most reasonable **and explain your reasoning**.



In [ ]:
# Your code here: compute thresholds (mean +/- k*SD) and plot them on the filtered signal


**Spike Waveforms and Clustering**

In the laptop college we covered how to detect spikes and extract their waveforms.  
Although full spike detection is **not required** for this course, you are welcome to try it yourself if you have extra time. It is a bit beyond the scope of the current practical, but it is a great way to deepen your understanding of neural data analysis.

To briefly summarize the idea:

1. Once a spike is detected at a certain timepoint, we can extract a short window of the signal around that moment (for example, a few milliseconds before and after).  
   This gives us the **waveform** of the spike.

2. Why extract spike waveforms?  
   Different neurons typically have **distinct spike shapes**, so waveform features can be used to separate spikes originating from different neurons.

3. A simple way to do this is by using **Principal Component Analysis (PCA)**.  
   PCA reduces the dimensionality of waveform data into components that capture most of the variance, allowing spikes to be visualized or clustered in a low-dimensional space.

4. Clustering the PCA features (for example, with k-means) can give you a **rough separation** of spikes into putative neurons.

However, this approach is a **very simplified version** of spike sorting.  
Modern neuroscience uses far more advanced algorithms such as **Kilosort**, which incorporate template matching, drift correction, GPU acceleration, and sophisticated noise modeling to achieve high-quality sorting on large-scale electrode arrays.

For this practical, we will **not** implement these advanced methods.  
Instead, you may explore PCA-based clustering as an optional challenge, but for the next exercise we will **provide you with pre-clustered neurons** so that everyone can continue the analysis from the same starting point.

Feel free to try the challenge if you're curious — otherwise, continue to the next exercise!


In [ ]:
# Optional: try spike detection and waveform clustering yourself here


**Loading the Clustered Neuron Data**

To help you get started, I have already clustered the spike waveforms into individual neurons.  
You do **not** need to perform spike sorting yourself for this practical.

You can now load the processed data directly.  
The dataset contains two key components:

1. **Spike times per neuron**  
   For each neuron, we provide the exact time points at which it fired.  

2. **A binary spike matrix**  
   This is an array of shape  
   $N_\text{neurons} \times N_\text{samples}$  
   where each entry is:
   - `0` → the neuron did not fire at that timepoint  
   - `1` → the neuron fired at that timepoint  
   



### **Plotting the Spike Pattern of a Neuron**

Below is an example of how you could visualize the spike pattern of a single neuron by plotting its spike times in a raster-like format. Change the code to do the plotting for different neurons and different time windows to get a feel for the data. 



In [ ]:
# Load the combined dictionary
data = np.load("spike_data.npy", allow_pickle=True).item()

t = data["time"]
spike_times = data["spike_times"]        # dict {neuron_id: times}
spike_matrix = data["spike_matrix"]      # binary matrix (neurons × samples
stim_times = data["stim_times"]
neuron_id = 2   # choose which neuron to plot

times = spike_times[neuron_id]

plt.figure(figsize=(10, 3))
plt.eventplot(times, colors='black', lineoffsets=1, linelengths=0.8)
plt.xlabel("Time (s)")
plt.yticks([])
plt.xlim([0, 1])
plt.title(f"Spike times for neuron {neuron_id}")
plt.show()


**Exercise — Creating a Trial-Aligned 3D Spike Matrix**

Now we can finally start preparing our data for real analysis! A general rule in neuroscience is to first convert recordings into a format that is easy to work with. In this experiment, we recorded **30 trials** of the same stimulus. A convenient way to represent this data is to create a **3D matrix** with dimensions:

$ \text{neurons} \times \text{timepoints} \times \text{trials} $

Each entry in this matrix indicates whether a given neuron fired (1) or not (0) at a particular timepoint during a specific trial.

Before building this matrix, we must decide on the **time window of interest** around each stimulus. This depends on when we expect neural responses to occur. One important rule is to always include some **baseline** time before stimulus onset so we have a comparison. A reasonable starting choice is:

$ -0.5\,\text{s} \rightarrow +0.5\,\text{s} $

but you are free to adjust this window depending on what you find most informative.

**Your task:**

Create a 3D NumPy matrix containing the spike responses aligned to each stimulus presentation.

- The matrix should have the shape:  
  $ (\text{neurons}, \text{timepoints}, \text{trials}) $
- Use your chosen time window.
- Use the binary spike matrix (`spike_matrix`, shaped as neurons × samples) to efficiently fill in the data.


In [ ]:
# Your code here: build the neurons x timepoints x trials matrix (window -0.5 to 0.5 s)
window_start = -0.5   # seconds before stimulus
window_end   =  0.5   # seconds after stimulus


**Exercise — Creating a Raster Plot From the Trial Matrix**

Now that our data is in a convenient format, we can start visualizing neural activity across trials. One common way to do this is with a **raster plot**, which shows the spike times for a neuron on each trial. You will generate this plot yourself using the steps below.

**Instructions**

1. Extract the **timepoints × trials** slice of the trial matrix for a chosen neuron.  
   This gives you the spike train across all trials for that neuron.

2. For each trial, find the indices where the neuron fired a spike  
   (these positions contain a value of `1` in the matrix).

3. Convert these spike indices into actual spike times using the time axis  
   (you can use `trial_times` for this).

4. Build a list that stores the spike times for each trial.  
   Each entry in the list should contain the spike times for one trial.

5. Create the raster plot.  
   You can use `matplotlib.pyplot.eventplot()` and pass it your list of spike-time arrays.  
   This will automatically draw one row per trial.

6. Add a line at t=0 to indicate the onset of the trial

Happy coding!

### **Questions to Answer**

- What do you observe in the rasterplot?
- Does one of the neurons show a clear increase in firing rate after the stimulus?  
- Are there neurons that do **not** respond to the stimulus at all?  



Happy coding!

In [ ]:
# Your code here: raster plot for one neuron


**Exercise — Creating a Peri-Stimulus Time Histogram (PSTH)**

Now that you have the raster plot, the next step is to quantify how each neuron responds to the stimulus. One common way to do this is by computing a **Peri-Stimulus Time Histogram (PSTH)**. A PSTH estimates the firing rate of a neuron as a function of time relative to a stimulus.

The basic idea:

- Divide the time window around the stimulus into small bins (for example, 10 ms).
- Count how many spikes the neuron fired in each bin.
- Convert spike counts into a firing rate estimate by dividing by  
  $ \text{binsize} \times \text{number of trials} $.
- Plot the firing rate across time to observe how the neuron responds.

A PSTH is essentially a smoothed, trial-averaged version of the raster plot.


**Instructions**

1. **Choose a neuron**  
   Select the index of the neuron you want to analyze.

2. **Extract its trial-aligned spike matrix**  
   From the 3D trial matrix (neurons × timepoints × trials), select the slice  
   corresponding to your chosen neuron.  
   This gives you a matrix of shape:  
   $ \text{timepoints} \times \text{trials} $.

3. **Define your bin size**  
   Choose a bin width (e.g., 10 ms).  
   Convert this to a number of samples using the sampling rate.

4. **Bin the spikes**  
   For each time bin:
   - Count all spikes across trials that fall into that bin.
   - Store the spike count in an array.

5. **Convert spike counts into firing rate**  
   For each bin compute:  
   $$
   \text{firing rate} = \frac{\text{spike count}}{\text{binsize (in seconds)} \times \text{number of trials}}
   $$

6. **Generate the PSTH plot**  
   - Use the midpoint of each time bin as the x-axis.  
   - Plot the firing rate as a function of time.  
   - Mark stimulus onset at time $0$.


### **Questions to Answer**

- What do you observe in the PSTH?  
- Does one of the neurons show a clear increase in firing rate after the stimulus?  
- Are there neurons that do **not** respond to the stimulus at all?  
- How does the PSTH compare to the raster plot you made earlier?
- How does the bin size effect the outcome?



Happy coding!


In [ ]:
# Your code here: PSTH (10 ms bins)


**Exercise — Statistical Comparison of Pre- vs Post-Stimulus Firing**

Now that we can compute firing rates, we want to know whether each neuron fires
more after the stimulus than before it. We will perform a simple **independent-
samples t-test**.

For each neuron:

1. Extract the spike counts from the **pre-stimulus window**  
   (e.g. −0.5 s to 0 s) across all trials.

2. Extract the spike counts from the **post-stimulus window**  
   (0 s to +0.5 s) across all trials.

3. Convert these spike counts into **firing rates per trial**  
   (spikes / window duration).

4. Perform a statistical test comparing the two sets of firing rates.

5. Plot the average firing rate before and after the stimulus for each neuron. (optionally mark significant neurons)



In [ ]:
# Your code here: pre- vs post-stimulus firing rates and t-test per neuron


#### Optional statistical sidenote
Technically speaking, we are doing multiple statistical tests here (namely, _n_ tests [for _n_ neurons]) so we have to correct for that. This is the multiple comparisons problem: since each test has a chance of returning a false positive (FP) according to the alpha value (in this case, 5% with alpha = 0.05), with multiple tests, the chance of getting a false positive becomes larger.

To show this mathematically, we can calculate the chance of getting more than one false positive as 1 minus the chance of getting no false positives. Since we run _n_ tests, this chance must be raised to the power _n_ (each test is separate, so for two tests, the chance of no FPs would be $P(\text{no FPs}) * P(\text{no FPs}) = P(\text{no FPs})^2$, for three tests it would be $P(\text{no FPs}) * P(\text{no FPs}) * P(\text{no FPs}) = P(\text{no FPs})^3$ etc.)

$$
P(\text{more than one FP}) = 1 - P(\text{no FPs})^n
$$

The probability of getting _no_ false positives is equivalent to one minus the probability of getting one. As we saw before, the probability of getting a false positive is equivalent to the alpha rate, so we get:

$$
P(\text{more than one FP}) = 1 - (1-\alpha)^n
$$

Let's consider a case with $\alpha = 0.05$ and for one test ($n=1$) and five tests ($n=5$):

$$
\begin{align}
n = 1 &\rightarrow P(\text{more than one FP}) = 1 - (1-\alpha)^n = 1 - (1-0.05)^1 = 1 - 0.95 = 0.05 \\
n = 5 &\rightarrow P(\text{more than one FP}) = 1 - (1-\alpha)^n = 1 - (1-0.05)^5 = 1 - 0.95^5 \approx 0.23
\end{align}
$$

So, for one test, indeed our false positive rate is neatly controlled with a chance of 5% - as we wanted. However, for 5 tests, the probability of getting a false positive is **23%**!

*Question: what would be the probability of getting a false positive in the tests we've just done on pre- vs post-stimulus firing rates?*

So how do we deal with this? One possibility is to correct our alpha (significance) level using the Bonferroni method. This is a fairly simple correction where the alpha level is divided by the number of tests:

$$
\alpha_\text{corr} = \frac{\alpha}{n}
$$

This means the significance threshold becomes "harder to reach". If we use the corrected alpha level, we can see that our false positive rate is controlled again to our original alpha level of 0.05:

$$
n=5 \rightarrow P(\text{more than one FP}) = 1 - (1 - \alpha)^n = 1 - \left(1 - \left(\frac{0.05}{5}\right)^5\right) = 1 - (0.99)^5 \approx 0.05
$$

_Question: Would the neurons that were significant in our t-tests still be significant with a corrected alpha level?_